# LucaVirus reference-free protein ranking

使用 LucaVirus 的蛋白 `prot_mask` MLM head，对每条序列逐有效氨基酸位点 mask，计算 `mean(log P(true_token | masked_sequence))` 并排序。当前是 H1 演示 FASTA；之后只需替换为淀粉酶多记录 FASTA。

In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'src' else Path.cwd().resolve()
LUCA_ROOT = Path('/home/chenyh/workspace/LucaVirusTasks')
FASTA_PATH = PROJECT_ROOT / 'src/ha_unaglined.fasta'  # 输入蛋白 FASTA；后续改为淀粉酶 FASTA
MODEL_PATH = LUCA_ROOT / 'llm/models/lucavirus/v1.0/token_level,span_level,seq_level/lucavirus/20240815023346/checkpoint-step3800000'
LOG_PATH = LUCA_ROOT / 'llm/logs/lucavirus/v1.0/token_level,span_level,seq_level/lucavirus/20240815023346/logs.txt'
OUTPUT_PATH = PROJECT_ROOT / 'outputs/bert_reference_free_ranking.csv'
MASK_BATCH_SIZE = 128  # 按显存调整；每个样本只 mask 一个氨基酸
DEVICE = torch.device('cuda:0')

if not FASTA_PATH.is_file() or not MODEL_PATH.is_dir() or not LOG_PATH.is_file():
    raise FileNotFoundError('请检查 FASTA、LucaVirus checkpoint 和 logs.txt 路径。')
sys.path[:0] = [str(LUCA_ROOT), str(LUCA_ROOT / 'src'), str(LUCA_ROOT / 'src/llm/lucavirus')]
from llm.lucavirus.get_embedding import load_model

_, _, model, tokenizer = load_model(str(LOG_PATH), str(MODEL_PATH), embedding_inference=False)
model = model.to(DEVICE).half().eval()  # 外层 FP16 推理；不修改 LucaVirusTasks

/home/chenyh/miniconda3/envs/fluProfiler/lib/python3.10/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/chenyh/miniconda3/envs/fluProfiler/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


import apex err: No module named 'apex'
------LLM Model Info ------
Model dirpath: /home/chenyh/workspace/LucaVirusTasks/llm/models/lucavirus/v1.0/token_level,span_level,seq_level/lucavirus/20240815023346/checkpoint-step3800000
Alphabet, vocab path: /home/chenyh/workspace/LucaVirusTasks/llm/models/lucavirus/v1.0/token_level,span_level,seq_level/lucavirus/20240815023346/checkpoint-step3800000/tokenizer
Load pretrained model: /home/chenyh/workspace/LucaVirusTasks/llm/models/lucavirus/v1.0/token_level,span_level,seq_level/lucavirus/20240815023346/checkpoint-step3800000
token_level/gene_mask
token_level/prot_mask
span_level/gene_type
span_level/prot_homo
span_level/prot_site
span_level/prot_domain
seq_level/gene_taxonomy
seq_level/prot_taxonomy
seq_level/prot_keyword


/home/chenyh/miniconda3/envs/fluProfiler/lib/python3.10/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [2]:
AA = set('ACDEFGHIKLMNPQRSTVWY')


def read_fasta(path):
    """Read a multiline FASTA; only a line-start '>' is a record header."""
    records, empty_headers = [], []
    header, sequence_lines = None, []

    def flush_record():
        if header is None:
            return
        sequence = ''.join(sequence_lines).upper()
        if not sequence:
            empty_headers.append(header)
            return
        sequence_id = header.split()[0] or f'sequence_{len(records) + 1}'
        records.append((sequence_id, sequence))

    with path.open(encoding='utf-8-sig') as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith('>'):
                flush_record()
                header, sequence_lines = line[1:].strip(), []
            elif header is None:
                raise ValueError(f'{path}: line {line_number} appears before the first FASTA header (>).')
            else:
                sequence_lines.append(''.join(line.split()))
    flush_record()

    if not records:
        raise ValueError(f'{path}: no non-empty FASTA records were found.')
    if empty_headers:
        examples = ', '.join(empty_headers[:3])
        raise ValueError(f'{path}: {len(empty_headers)} empty FASTA record(s), e.g. {examples}')
    return records

def encode(sequence):
    tokens = tokenizer.encode(sequence)
    valid = [i + int(tokenizer.prepend_bos) for i, residue in enumerate(sequence)
             if residue in AA and tokens[i] != tokenizer.unk_idx]
    ids = torch.tensor([tokenizer.cls_idx, *tokens, tokenizer.eos_idx], dtype=torch.long)
    return ids, valid

def position_diversity(records: list[tuple[str, str]]):
    values = [seq for _, seq in records]

    if not values:
        return [], []

    if len({len(s) for s in values}) != 1:
        raise ValueError("所有字符串必须等长")

    diversity = [len(set(chars)) for chars in zip(*values)]
    unchanged_positions = [i for i, n in enumerate(diversity) if n == 1]

    return diversity, unchanged_positions

records = read_fasta(FASTA_PATH)
print(f'已读取 {len(records):,} 条。')
diversity, unchanged = position_diversity(records)

print("diversity:", diversity)  # [1, 2, 1, 2]
print("unchanged:", unchanged)  # [0, 2]

已读取 34,358 条。
diversity: [11, 13, 14, 15, 11, 12, 11, 9, 11, 8, 11, 10, 11, 13, 10, 14, 6, 6, 9, 7, 4, 7, 6, 7, 10, 5, 8, 5, 6, 5, 8, 8, 8, 7, 7, 7, 6, 6, 8, 7, 7, 9, 6, 3, 8, 8, 9, 9, 7, 7, 7, 11, 9, 6, 9, 6, 6, 3, 5, 11, 6, 12, 7, 13, 10, 5, 6, 11, 4, 10, 12, 3, 9, 6, 9, 5, 4, 6, 5, 4, 7, 4, 9, 4, 8, 11, 12, 12, 11, 11, 12, 7, 6, 7, 5, 7, 9, 5, 12, 10, 12, 6, 13, 9, 4, 10, 4, 5, 4, 4, 12, 4, 15, 10, 5, 6, 5, 4, 6, 7, 7, 7, 6, 9, 7, 8, 6, 8, 7, 11, 7, 5, 7, 7, 8, 12, 12, 10, 7, 6, 9, 14, 8, 11, 11, 12, 15, 6, 11, 5, 13, 4, 4, 11, 12, 13, 6, 15, 11, 7, 5, 6, 11, 7, 7, 8, 5, 7, 6, 11, 9, 11, 15, 12, 6, 10, 12, 9, 11, 14, 10, 5, 11, 4, 13, 8, 7, 10, 11, 5, 5, 7, 9, 4, 6, 6, 6, 7, 5, 8, 7, 16, 15, 13, 4, 17, 13, 6, 3, 11, 10, 11, 13, 7, 8, 5, 10, 5, 10, 5, 8, 11, 6, 10, 9, 10, 11, 13, 8, 8, 8, 13, 13, 4, 9, 9, 7, 8, 11, 5, 10, 7, 3, 7, 8, 3, 6, 4, 9, 8, 7, 12, 11, 8, 7, 12, 7, 9, 4, 8, 5, 7, 5, 4, 6, 8, 6, 6, 11, 5, 5, 6, 13, 8, 11, 10, 15, 14, 12, 9, 8, 7, 7, 11, 7, 9, 12, 12, 11, 11, 11

In [ ]:
def score(records):
    totals, valid_lengths, jobs = (torch.zeros(len(records), dtype=torch.float64),
                                   torch.zeros(len(records), dtype=torch.long), [])
    total_positions = sum(sum(residue in AA for residue in sequence) for _, sequence in records)
    progress = tqdm(total=total_positions, desc='BERT 打分', unit='aa')

    def flush():
        if not jobs:
            return
        input_ids = pad_sequence([ids for _, ids, _ in jobs], batch_first=True,
                                 padding_value=tokenizer.padding_idx)
        rows = torch.arange(len(jobs))
        positions = torch.tensor([position for _, _, position in jobs])
        targets = input_ids[rows, positions].clone()
        input_ids[rows, positions] = tokenizer.mask_idx
        with torch.inference_mode():
            output = model(
                input_ids_b=input_ids.to(DEVICE),
                token_type_ids_b=input_ids.ne(tokenizer.padding_idx).long().to(DEVICE),
                output_keys_b={'token_level': {'prot_mask'}},
                need_head_weights=False, repr_layers=[], return_dict=True,
            )
            logits = output.outputs_b['token_level']['prot_mask']
            log_probs = logits.float().log_softmax(-1)[rows.to(DEVICE), positions.to(DEVICE), targets.to(DEVICE)]
        totals.index_add_(0, torch.tensor([index for index, *_ in jobs]), log_probs.cpu().to(torch.float64))
        progress.update(len(jobs))
        jobs.clear()

    for index, (_, sequence) in enumerate(records):
        ids, positions = encode(sequence)
        valid_lengths[index] = len(positions)
        for position in positions:
            jobs.append((index, ids, position))
            if len(jobs) == MASK_BATCH_SIZE:
                flush()
    flush()
    progress.close()
    if (valid_lengths == 0).any():
        raise ValueError('至少一条序列没有可打分的标准氨基酸位点。')
    return totals / valid_lengths, valid_lengths


scores, valid_lengths = score(records)
ranking = pd.DataFrame(records, columns=['sequence_id', 'sequence'])
ranking['valid_length'], ranking['bert_score'] = valid_lengths.numpy(), scores.numpy()
ranking = ranking.sort_values('bert_score', ascending=False, kind='stable').reset_index(drop=True)
ranking['rank'] = ranking.index + 1
ranking = ranking[['sequence_id', 'sequence', 'valid_length', 'bert_score', 'rank']]
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
ranking.to_csv(OUTPUT_PATH, index=False)
display(ranking)

BERT 打分:   0%|          | 9216/19411627 [01:50<64:58:19, 82.95aa/s]

KeyboardInterrupt: 

BERT 打分:   0%|          | 9216/19411627 [02:10<64:58:19, 82.95aa/s]